# Import Libraries

In [1]:
#!pip install BlackBoxAuditing 

In [2]:
#!pip install aif360

In [3]:
#!pip install tensorflow

In [4]:
#!pip install opencv-python

In [5]:
#!pip install fairlearn

In [6]:
#!pip install --upgrade aif360

In [7]:
import tensorflow as tf
tf.compat.v1.enable_eager_execution()

# Verify it's enabled
print("Eager execution:", tf.executing_eagerly())  # Must print True

Eager execution: True


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
tf.compat.v1.enable_eager_execution()
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score
import keras
from keras.layers import *
from keras.models import *
from keras.optimizers import Adam
from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric

from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from aif360.algorithms.postprocessing import CalibratedEqOddsPostprocessing
from aif360.datasets import StandardDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from tqdm import tqdm

import warnings
from pandas.errors import PerformanceWarning
from sklearn.exceptions import DataConversionWarning

# NOTE: `category` must be the exception CLASS, not its name as a string.
# warnings.simplefilter() does not validate this argument, so a string here
# lands in warnings.filters and every later warnings.warn() call dies with
#     TypeError: issubclass() arg 2 must be a class, a tuple of classes, ...
# because the filter loop evaluates issubclass(category, 'PerformanceWarning').
# That surfaced as a crash inside sklearn's LabelEncoder when the reductions
# tried to emit an ordinary DataConversionWarning.
warnings.simplefilter(action='ignore', category=PerformanceWarning)

# aif360 hands sklearn a column-vector y, which is what raised that warning.
# It is harmless, so silence it and keep the output readable.
warnings.filterwarnings('ignore', category=DataConversionWarning)

# Safety net: drop any filter already registered with a non-class category,
# e.g. left over in the kernel from an earlier run of this notebook.
_bad = [f for f in warnings.filters if not isinstance(f[2], type)]
for _f in _bad:
    warnings.filters.remove(_f)
if _bad:
    warnings._filters_mutated()
    print('removed', len(_bad), 'malformed warning filter(s):', _bad)


# Prepare Dataset

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

# ── Run settings ──────────────────────────────────────────────────────────────
# Every configuration is run once per random seed (MIDRC_SEED = 0 .. 4); the seed
# controls the CV folds, the network initialisation and the randomised reductions.
SEED    = int(os.environ.get("MIDRC_SEED", "0"))
P_COMP  = int(os.environ.get("MIDRC_PCA", "16"))      # PCA components of the image embedding
LIBRARY = "AIF360"

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# ── MIDRC data: real COVID-19 labels and real (self-reported) demographics ─────
DATA_CSV = os.environ.get("MIDRC_DATA_CSV", r"C:\midrc\dataset_final.csv")
FEATURES = os.environ.get("MIDRC_FEATURES", r"C:\midrc\features_xrv_densenet121.npz")
RESULTS_DIR = os.path.join(os.environ.get("MIDRC_RESULTS", os.path.join(os.getcwd(), "Results")),
                           f"seed_{SEED}", LIBRARY)
os.makedirs(RESULTS_DIR, exist_ok=True)

meta = pd.read_csv(DATA_CSV)
feat = np.load(FEATURES, allow_pickle=True)
emb  = pd.DataFrame(feat["embeddings"], index=feat["fname"])     # 1024-d embedding per image
meta = meta[meta["fname"].isin(emb.index)].reset_index(drop=True)
if os.environ.get("MIDRC_SMOKE"):                                 # optional quick test on a subset
    meta = meta.sample(int(os.environ["MIDRC_SMOKE"]), random_state=42).reset_index(drop=True)

# Status: COVID-19 test result (PCR / antigen) provided by MIDRC
meta['Status'] = np.where(meta['covid19_positive'] == 'Yes', 'Covid', 'Non-COVID')

# Gender: self-reported sex
meta['Gender'] = meta['sex']

# Ethnicity: self-reported race / ethnicity grouped into 5 categories
RACE_GROUPS = {'White': 'White', 'Black or African American': 'Black', 'Asian': 'Asian'}
def ethnicity_group(row):
    if row['ethnicity'] == 'Hispanic or Latino':
        return 'Hispanic'
    return RACE_GROUPS.get(row['race'], 'Others')
meta['Ethnicity'] = meta.apply(ethnicity_group, axis=1)

# Age at imaging (years); missing ages are filled with the median age
meta['Age'] = pd.to_numeric(meta['age_at_imaging'], errors='coerce')
meta['Age'] = meta['Age'].fillna(meta['Age'].median() if meta['Age'].notna().any() else 0).round().astype(int)

# Patient identifier: used to keep all images of a patient in the same CV fold
GROUPS = meta['case_id'].values


def grouped_cv(n_splits=5, **_):
    """Stratified K-fold grouped by patient: no patient appears in two folds.
    Replaces the StratifiedKFold of Version 2; the folds change with SEED."""
    return StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                random_state=SEED).split(X, y, GROUPS)


def calibration_split(train_index, cal_size=0.25):
    """Split a training fold (by patient) into a fitting part and a calibration part.
    Post-processors are fitted on the calibration part, never on the test fold."""
    gss = GroupShuffleSplit(n_splits=1, test_size=cal_size, random_state=SEED)
    fit_pos, cal_pos = next(gss.split(train_index, groups=GROUPS[train_index]))
    return np.asarray(train_index)[fit_pos], np.asarray(train_index)[cal_pos]


os.chdir(RESULTS_DIR)        # every file written below goes to Results/seed_<SEED>/<LIBRARY>
print(f"{LIBRARY} | seed {SEED} | {len(meta)} images | {meta['case_id'].nunique()} patients | results -> {RESULTS_DIR}")


# ---------------------------------------------------------------------------
# Early stopping without test-fold leakage.
# Every network below stops on val_loss computed on a random hold-out of
# ES_VAL_FRAC of the TRAINING rows of the current fold (seeded by SEED); the
# test fold is never seen during training.  Rows are permuted first because
# dataset_final.csv is sorted by label.
# ---------------------------------------------------------------------------
ES_VAL_FRAC = 0.2

def fit_es(model, X, y, sample_weight=None, validation_data=None, validation_split=None, **kw):
    X = np.asarray(X, dtype='float32')
    y = np.asarray(y)
    idx = np.random.RandomState(SEED).permutation(len(X))
    k = int(round(ES_VAL_FRAC * len(X)))
    va, tr = idx[:k], idx[k:]
    if sample_weight is None:
        return model.fit(X[tr], y[tr], validation_data=(X[va], y[va]), **kw)
    sw = np.asarray(sample_weight, dtype='float32').ravel()
    return model.fit(X[tr], y[tr], sample_weight=sw[tr],
                     validation_data=(X[va], y[va], sw[va]), **kw)


In [ ]:
print("images:", len(meta), "| patients:", meta['case_id'].nunique(), "| seed:", SEED)
print(meta["Status"].value_counts())

In [ ]:
print(pd.crosstab(meta['Ethnicity'], meta['Status'], margins=True))
print(pd.crosstab(meta['Gender'], meta['Status'], margins=True))

In [ ]:
print(meta.shape)
print(meta['Status'].value_counts())

In [ ]:
meta[['fname', 'Status', 'Age', 'Gender', 'Ethnicity', 'case_id']].head(10)

In [ ]:
meta[['fname', 'Status', 'Age', 'Gender', 'Ethnicity', 'case_id']].head(10)

In [ ]:
# Age, Gender and Ethnicity are the real MIDRC values loaded with the data above.
print(meta[['Age']].describe().T)
print(meta['Gender'].value_counts())
print(meta['Ethnicity'].value_counts())

In [ ]:
meta[['fname', 'Status', 'Age', 'Gender', 'Ethnicity', 'case_id']].head(10)

In [ ]:
meta[['fname', 'Status', 'Age', 'Gender', 'Ethnicity', 'case_id']].head(10)

In [ ]:
# Image normalisation is done by the feature extractor (C:\midrc\extract_features.py):
# each PNG is scaled to the [-1024, 1024] range expected by TorchXRayVision,
# centre-cropped and resized to 224 x 224.

# 1st Model: Feature Extraction (chest-X-ray-pretrained DenseNet-121)

In [ ]:
# Feature extractor: DenseNet-121 from TorchXRayVision (weights "densenet121-res224-all"),
# pretrained on ~800,000 chest radiographs from eight public datasets (NIH, CheXpert,
# MIMIC-CXR, PadChest, ...).  None of these is MIDRC, and the network is used frozen,
# so it never sees our labels; this is why no separate image train/test split is
# needed before the feature extraction step.
# The 1024-d penultimate-layer embedding of every image was computed once by
# C:\midrc\extract_features.py and is loaded in the data cell above (`emb`).
print("embedding:", emb.shape)

In [ ]:
# Features: real demographics + image embedding
x = meta[['Age', 'Gender', 'Ethnicity']].copy()
y = meta['Status'].copy()

In [ ]:
status_mapping = {'Non-COVID': 0, 'Covid': 1}
y = y.map(status_mapping)

In [ ]:
# Image embeddings (frozen, CXR-pretrained DenseNet-121), in the same order as `meta`
intermediate_output = emb.loc[meta['fname']].values.astype('float32')
intermediate_output.shape

## Image embeddings (1024-d, from the pretrained DenseNet-121)

In [ ]:
# (V2 extracted features from a CNN trained here; the embeddings are now pre-computed.)

In [ ]:
# (no image arrays are held in memory any more)

## Reducing the Dimensionality of the Image Embeddings

In [ ]:
from sklearn.decomposition import PCA
p_comp = P_COMP
pca = PCA(n_components=p_comp, random_state=SEED)
x[[f'PCA{i}' for i in range(p_comp)]] = pca.fit_transform(intermediate_output)
print(f"PCA: {p_comp} components, {pca.explained_variance_ratio_.sum():.1%} of the embedding variance")
del intermediate_output

# 2nd Model: Predicting COVID-19 Status (image features + Age, Gender and Ethnicity)

In [ ]:
X = x.reset_index(drop=True)
y = y.reset_index(drop=True)
assert len(X) == len(y) == len(GROUPS)

In [32]:
# Label Encoding
feats = ['Gender','Ethnicity']
le = LabelEncoder()
for f in feats:
    X[f] = le.fit_transform(X[f])

In [33]:
# ── Results store ───────────────────────────────────────────────────────────
# One tidy row per (Algorithm, Sensitive Feature, Status).
# Metrics stored: Accuracy, Accuracy Difference, Demographic Parity Difference,
# Equalized Odds Difference.
RESULT_COLUMNS = [
    'Algorithm', 'Sensitive Feature', 'Status',
    'Accuracy', 'Accuracy Difference',
    'Demographic Parity Difference', 'Equalized Odds Difference',
    'N Groups',
]

metrics_values = pd.DataFrame(columns=RESULT_COLUMNS)

In [34]:
# ═══════════════════════════════════════════════════════════════════════════
# FAIRNESS METRICS + RESULTS STORE
#
# Identical machinery to the Fairlearn notebook, so the two studies produce
# directly comparable tables. Every algorithm below calls evaluate_fairness(),
# which appends one tidy row per (Algorithm, Sensitive Feature, Status) to
# `metrics_values`; the final cell pivots that into the comparison tables and
# writes the CSV.
#
# Note this uses fairlearn's MetricFrame for the reporting even though the
# mitigation algorithms come from AIF360 - the metric definitions are the same
# either way, and reusing them keeps the two notebooks on the same footing.
# ═══════════════════════════════════════════════════════════════════════════
def multiclass_selection_rate(y_true, y_pred, desired_class):
    """Selection rate for one Status class in the multiclass problem.

    The selection rate is the fraction of predicted labels equal to `desired_class`
    (i.e. P[Y_hat = class] within the group). This is the quantity compared across
    groups by the demographic parity difference.

    Args:
        y_true : array-like  -- true labels (unused, kept for the MetricFrame signature)
        y_pred : array-like  -- predicted *labels* (not probabilities)
        desired_class : any  -- the Status class of interest
    Returns:
        float -- selection rate for the desired class
    """
    y_pred = np.asarray(y_pred)
    total = y_pred.shape[0]
    return np.nan if total == 0 else (y_pred == desired_class).sum() / total

from fairlearn.metrics import MetricFrame
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# ── Status classes (must match the mapping used for y) ──────────────────────
STATUS_MAPPING = {'Non-COVID': 0, 'Covid': 1}
CLASS_NAMES = {v: k for k, v in STATUS_MAPPING.items()}
N_CLASSES = len(STATUS_MAPPING)

# ── Sensitive-feature configurations to report on ───────────────────────────
SENSITIVE_SPECS = {
    'Gender': ['Gender'],
    'Ethnicity': ['Ethnicity'],
    'Gender + Ethnicity': ['Gender', 'Ethnicity'],
}

# Readable labels for the label-encoded columns
GENDER_LABELS = {0: 'Female', 1: 'Male'}
ETHNICITY_LABELS = {0: 'Asian', 1: 'Black', 2: 'Hispanic', 3: 'Others', 4: 'White'}


def to_labels(y_score, n_classes=N_CLASSES):
    """Convert a score/probability matrix to hard Status labels.

    The networks emit 4 output columns (a leftover from the age-group model), so
    only the first `n_classes` columns - the ones that correspond to actual Status
    values - are considered. A 1-D input is assumed to already be labels.
    """
    arr = np.asarray(y_score.values if hasattr(y_score, 'values') else y_score)
    if arr.ndim == 1:
        return arr.astype(int)
    return np.argmax(arr[:, :n_classes], axis=1).astype(int)


def build_sensitive_feature(X, cols):
    """Return a single Series of (possibly intersectional) group labels."""
    if len(cols) == 1:
        col = cols[0]
        s = pd.Series(np.asarray(X[col]), name=col)
        if col == 'Gender':
            s = s.map(lambda v: GENDER_LABELS.get(v, v))
        elif col == 'Ethnicity':
            s = s.map(lambda v: ETHNICITY_LABELS.get(v, v))
        return s.astype(str)
    parts = [build_sensitive_feature(X, [c]) for c in cols]
    combined = parts[0].str.cat(parts[1:], sep=' | ')
    combined.name = ' + '.join(cols)
    return combined


# ── Per-group rates for a one-vs-rest (binary) view of a Status class ───────
def _selection_rate(y_true, y_pred):
    y_pred = np.asarray(y_pred)
    return np.nan if len(y_pred) == 0 else float((y_pred == 1).mean())


def _tpr(y_true, y_pred):
    """True positive rate; NaN when the group has no positive cases."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    pos = (y_true == 1).sum()
    return np.nan if pos == 0 else float(((y_pred == 1) & (y_true == 1)).sum() / pos)


def _fpr(y_true, y_pred):
    """False positive rate; NaN when the group has no negative cases."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    neg = (y_true == 0).sum()
    return np.nan if neg == 0 else float(((y_pred == 1) & (y_true == 0)).sum() / neg)


BINARY_METRICS = {
    'accuracy': lambda yt, yp: accuracy_score(np.asarray(yt), np.asarray(yp)),
    'selection_rate': _selection_rate,
    'tpr': _tpr,
    'fpr': _fpr,
    'count': lambda yt, yp: len(yt),
}


def _spread(values):
    """max - min across groups, ignoring groups where the rate is undefined."""
    v = pd.to_numeric(pd.Series(list(values)), errors='coerce').dropna()
    return np.nan if len(v) < 2 else float(v.max() - v.min())


# ── Metrics used by the by-group tables (now named after Status) ────────────
metrics = {
    'accuracy': lambda y_true, y_pred: accuracy_score(y_true, to_labels(y_pred)),
    'selection_rate_Non-COVID': lambda y_true, y_pred: multiclass_selection_rate(y_true, to_labels(y_pred), STATUS_MAPPING['Non-COVID']),
    'selection_rate_Covid':     lambda y_true, y_pred: multiclass_selection_rate(y_true, to_labels(y_pred), STATUS_MAPPING['Covid']),
    'count': lambda y_true, y_pred: len(y_true),
}


def evaluate_fairness(y_true, y_score, X, algorithm, store=True, verbose=True, decimals=3):
    """Accuracy, demographic parity difference and equalized odds difference,
    computed for every Status value and every sensitive-feature configuration.

    For each Status class the problem is binarised one-vs-rest, then:
      * Accuracy                      -- accuracy of that one-vs-rest decision
                                         (the 'Overall (all classes)' row is the
                                         plain multiclass accuracy)
      * Accuracy Difference           -- max - min accuracy across groups
      * Demographic Parity Difference -- max - min selection rate across groups
      * Equalized Odds Difference     -- max(TPR spread, FPR spread) across groups

    Groups where a rate is undefined (no positives / no negatives) are skipped
    rather than raising, so small intersectional groups can't break the run.
    """
    global metrics_values

    y_labels = to_labels(y_score)
    y_true_arr = np.asarray(y_true.values if hasattr(y_true, 'values') else y_true).astype(int)

    rows = []
    for sf_name, cols in SENSITIVE_SPECS.items():
        sf = build_sensitive_feature(X, cols)

        # Multiclass accuracy, overall and by group
        mf_all = MetricFrame(
            metrics={'accuracy': lambda yt, yp: accuracy_score(np.asarray(yt), np.asarray(yp))},
            y_true=y_true_arr, y_pred=y_labels, sensitive_features=sf,
        )
        rows.append({
            'Algorithm': algorithm,
            'Sensitive Feature': sf_name,
            'Status': 'Overall (all classes)',
            'Accuracy': accuracy_score(y_true_arr, y_labels),
            'Accuracy Difference': _spread(mf_all.by_group['accuracy']),
            'Demographic Parity Difference': np.nan,
            'Equalized Odds Difference': np.nan,
            'N Groups': int(mf_all.by_group.shape[0]),
        })

        # One row per Status value
        for cls_val in sorted(CLASS_NAMES):
            yt = (y_true_arr == cls_val).astype(int)
            yp = (y_labels == cls_val).astype(int)
            mf = MetricFrame(metrics=BINARY_METRICS, y_true=yt, y_pred=yp, sensitive_features=sf)
            rows.append({
                'Algorithm': algorithm,
                'Sensitive Feature': sf_name,
                'Status': CLASS_NAMES[cls_val],
                'Accuracy': accuracy_score(yt, yp),
                'Accuracy Difference': _spread(mf.by_group['accuracy']),
                'Demographic Parity Difference': _spread(mf.by_group['selection_rate']),
                'Equalized Odds Difference': np.nanmax([
                    _spread(mf.by_group['tpr']),
                    _spread(mf.by_group['fpr']),
                ]),
                'N Groups': int(mf.by_group.shape[0]),
            })

    results = pd.DataFrame(rows, columns=RESULT_COLUMNS)

    if store:
        if len(metrics_values):
            base = metrics_values[metrics_values['Algorithm'] != algorithm]
            metrics_values = (
                pd.concat([base, results], ignore_index=True)
                if len(base) else results.copy()
            )
        else:
            metrics_values = results.copy()

    if verbose:
        print(f'=== {algorithm} ===')
        print(results.round(decimals).to_string(index=False))

    return results


def group_breakdown(y_true, y_score, X, sensitive='Gender', decimals=3):
    """Per-group accuracy / selection rate / TPR / FPR for each Status value.

    These are the raw numbers the differences above are computed from - useful for
    seeing *which* group drives a large disparity.
    """
    y_labels = to_labels(y_score)
    y_true_arr = np.asarray(y_true.values if hasattr(y_true, 'values') else y_true).astype(int)
    sf = build_sensitive_feature(X, SENSITIVE_SPECS[sensitive])

    tables = {}
    for cls_val in sorted(CLASS_NAMES):
        yt = (y_true_arr == cls_val).astype(int)
        yp = (y_labels == cls_val).astype(int)
        mf = MetricFrame(metrics=BINARY_METRICS, y_true=yt, y_pred=yp, sensitive_features=sf)
        tables[CLASS_NAMES[cls_val]] = mf.by_group

    return pd.concat(tables, names=['Status']).round(decimals)


def comparison_table(metric='Equalized Odds Difference', decimals=3):
    """Wide Algorithm-by-(Sensitive Feature, Status) table for one metric."""
    if not len(metrics_values):
        return metrics_values
    return (metrics_values
            .pivot_table(index=['Sensitive Feature', 'Status'],
                         columns='Algorithm', values=metric, sort=False)
            .round(decimals))

In [ ]:
# ── AIF360 metric helper ─────────────────────────────────────────────────────
# One function replaces the 24 copies of the metric loop in Version 2; the
# definition is unchanged: maximum of each metric over favourable class,
# privileged class and privileged group (the other listed groups unprivileged).
from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric

ETH_CODES = [0.0, 1.0, 2.0, 4.0]          # Asian, Black, Hispanic, White (Others excluded, as in V2)
AIF_SETTINGS = {
    'Gender':           (['Gender'], [[[0]], [[1]]], [{'Gender': 0}, {'Gender': 1}]),
    'Ethnicity':        (['Ethnicity'], [[[e]] for e in ETH_CODES], [{'Ethnicity': e} for e in ETH_CODES]),
    'Gender+Ethnicity': (['Gender', 'Ethnicity'],
                         [[[g], [e]] for g in [0, 1] for e in ETH_CODES],
                         [{'Gender': g, 'Ethnicity': e} for g in [0, 1] for e in ETH_CODES]),
}


def _safe(fn):
    try:
        v = fn()
        return v if np.isfinite(v) else 0.0
    except (TypeError, ZeroDivisionError, FloatingPointError):
        return 0.0


def aif_metrics(pred, attr):
    """Max of each AIF360 metric over favourable class x privileged class x group."""
    names, priv_lists, groups = AIF_SETTINGS[attr]
    labels = to_labels(pred)
    best = dict.fromkeys(['SPD', 'AOD', 'EOD', 'Theil', 'GEI', 'DI'], -np.inf)
    for fav in range(N_CLASSES):
        for priv in priv_lists:
            ds = StandardDataset(df=pd.concat([X, y.rename('Status')], axis=1), label_name='Status',
                                 favorable_classes=[fav], protected_attribute_names=names,
                                 privileged_classes=priv)
            ds.labels = ds.labels.astype(int)
            pds = ds.copy()
            pds.labels = labels.astype(float).reshape(-1, 1)
            for i, g in enumerate(groups):
                m = ClassificationMetric(ds, pds, unprivileged_groups=groups[:i] + groups[i + 1:],
                                         privileged_groups=[g])
                for k, f in [('SPD', m.statistical_parity_difference), ('AOD', m.average_odds_difference),
                             ('EOD', m.equal_opportunity_difference), ('Theil', m.theil_index),
                             ('GEI', m.generalized_entropy_index), ('DI', m.disparate_impact)]:
                    best[k] = max(best[k], _safe(f))
    return best


def aif_report(pred, attr):
    b = aif_metrics(pred, attr)
    print(f"{attr} Bias Identification:")
    print("Max Statistical Parity Difference (Ideal value = 0):", b['SPD'])
    print("Max Average Odds Difference (Ideal value = 0):", b['AOD'])
    print("Max Equal Opportunity Difference (Ideal value = 0):", b['EOD'])
    print("Max Theil Index (Ideal value = 0):", b['Theil'])
    print("Max Generalized Entropy Index (Ideal value = 0):", b['GEI'])
    print("Max Disparate Impact (Ideal value = 1):", b['DI'])
    return b


# Original Model

In [ ]:
# Model
inputs_dense = Input(shape=(X.shape[1],))
x = Dense(8, activation='relu', kernel_initializer='glorot_uniform')(inputs_dense)
#x = Dropout(0.5)(x)
x = Dense(6, activation='relu', kernel_initializer='glorot_uniform')(x)
outputs_dense = Dense(4, activation='softmax', kernel_initializer='glorot_uniform')(x)
dense_model = Model(inputs=inputs_dense, outputs=outputs_dense)

# Save Initial Weights
initial_weights = dense_model.get_weights()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# CV and OOF Predictions
y_pred = pd.DataFrame(0.0,columns=[0,1,2,3],index=X.index)

scores = []                   
for train_index, test_index in grouped_cv(n_splits=5):
    X_Train, X_Test = X.iloc[train_index,:], X.iloc[test_index,:]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
    
    inputs_dense = Input(shape=(X.shape[1],))
    x = Dense(8, activation='relu', kernel_initializer='glorot_uniform')(inputs_dense)
    x = Dense(6, activation='relu', kernel_initializer='glorot_uniform')(x)
    outputs_dense = Dense(4, activation='softmax', kernel_initializer='glorot_uniform')(x)
    dense_model = Model(inputs=inputs_dense, outputs=outputs_dense)
    dense_model.compile(optimizer=Adam(learning_rate=0.01), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    # Reinitialize the weights
    dense_model.set_weights(initial_weights)
    
    # Define early stopping
    es = EarlyStopping(monitor='val_loss', mode='min',patience=3)

    fit_es(dense_model, X_Train.values.astype('float32'), y_Train.values.astype('float32'),
                    epochs=25, batch_size=64, verbose=0,
                    callbacks=[es])  
    preds = dense_model(X_Test.values.astype('float32')).numpy().squeeze()
    y_pred.iloc[test_index,:] = preds
    
    scores.append(accuracy_score(y_Test.values.astype('float32'),np.argmax(preds,axis=1)))
    print(scores[-1])

print("Mean:",np.mean(scores),"\nSTD: ", np.std(scores),'\n')


# 0.7342332841172748
# 0.7449905083315757
# 0.7333895802573297
# 0.7308584686774942
# 0.7373971735920691
# Mean: 0.7361738029951488 
# STD:  0.004878549318893849 

In [ ]:
# Save the out-of-fold predictions of the Original Model and the evaluation frame
# (labels, sensitive attributes, patient id) for the cross-seed analysis notebook.
y_pred.to_csv('y_pred_Original.csv', index=False)
pd.DataFrame({'Status': y.values, 'Age': X['Age'].values, 'Gender': X['Gender'].values,
              'Ethnicity': X['Ethnicity'].values, 'case_id': GROUPS}).to_csv('eval_frame.csv', index=False)

In [38]:
# ── Fairness metrics for Original (appended to metrics_values) ──
results_original = evaluate_fairness(y, y_pred, X, algorithm='Original')
results_original


=== Original ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
 Original             Gender Overall (all classes)     0.595                0.012                            NaN                        NaN         2
 Original             Gender             Non-COVID     0.595                0.012                          0.019                      0.032         2
 Original             Gender                 Covid     0.595                0.012                          0.019                      0.032         2
 Original          Ethnicity Overall (all classes)     0.595                0.148                            NaN                        NaN         5
 Original          Ethnicity             Non-COVID     0.595                0.148                          0.628                      0.635         5
 Original          Ethnicity                 Covid     0.595                0.148  

# Metrics

In [ ]:
# AIF360 metrics, Gender (computed with the shared helper `aif_report`)
_ = aif_report(y_pred, 'Gender')

In [ ]:
# AIF360 metrics, Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred, 'Ethnicity')

In [ ]:
# AIF360 metrics, Gender+Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred, 'Gender+Ethnicity')

## Shared model architecture

The mitigation sections below were each defining their own network (`Input(21)` with
16 -> 8 units, or 128 -> 64 -> 32 units, Adam at its default learning rate, dropout,
no early stopping). The cell below extracts the architecture of the **Original Model**
into one factory so that every section trains the same network and the accuracies stay
comparable.


In [ ]:
import os
# ===========================================================================
# Shared architecture for every model in the notebook
#
# Taken verbatim from the "Original Model" section (mean CV accuracy 0.8076):
#
#     Input(n_features) -> Dense(8, relu) -> Dense(6, relu) -> Dense(n_out)
#     Adam(learning_rate=0.01) | 25 epochs | batch_size 64 | no dropout
#     EarlyStopping(monitor='val_loss', mode='min', patience=3)
#     the same starting weights are reused for every fold
#
# Every mitigation section below now builds its network through this factory,
# so any difference in accuracy comes from the mitigation algorithm and not
# from a different network. Two heads are provided because the notebook uses
# both shapes:
#     n_outputs=4 -> softmax + sparse_categorical_crossentropy (multiclass)
#     n_outputs=1 -> sigmoid + binary_crossentropy (one-vs-rest sections)
# ===========================================================================
import numpy as np
import tensorflow as _tf   # own alias: later cells rebind `tf` to compat.v1
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.base import BaseEstimator, ClassifierMixin

ORIG_H1, ORIG_H2 = 8, 6          # hidden widths of the original model
ORIG_LR          = 0.01
ORIG_EPOCHS      = 25
ORIG_BATCH_SIZE  = 64
ORIG_PATIENCE    = 3

# Budget for the two fairlearn reductions used in the in-processing and
# sequential sections. Each retrains the network once per iteration / grid
# point, so the library defaults (max_iter=50, grid_size=10) are expensive
# here. Raise these if you have the runtime to spare.
EGR_MAX_ITER     = 10   # library default: 50
GSR_GRID_SIZE    = 5    # library default: 10
N_FEATURES       = X.shape[1]    # read from the data - never hard-code 11/21


if os.environ.get("MIDRC_FAST"):  # quick test runs only
    ORIG_EPOCHS, EGR_MAX_ITER, GSR_GRID_SIZE = 2, 2, 2

# ── Helpers for the post-processing methods (Version 3) ──────────────────────
def gender_dataset(X_part, labels, scores=None):
    """Binary AIF360 dataset with Gender as protected attribute. Labels are 0/1
    decisions; model probabilities go into .scores (Version 2 put probabilities
    into the label column, which AIF360 then binarised to all-unfavourable)."""
    ds = StandardDataset(
        pd.concat([X_part[['Gender']].reset_index(drop=True),
                   pd.DataFrame(np.asarray(labels).astype(int).ravel(), columns=['Status'])], axis=1),
        label_name='Status', favorable_classes=[1],
        protected_attribute_names=['Gender'], privileged_classes=[[1]])
    if scores is not None:
        ds.scores = np.asarray(scores, dtype=float).reshape(-1, 1)
    return ds

_INITIAL_WEIGHTS = {}            # one fixed init per (input_dim, n_outputs)


def build_original_arch(input_dim=None, n_outputs=4):
    """Compiled copy of the original architecture, reset to shared init weights."""
    input_dim = N_FEATURES if input_dim is None else int(input_dim)
    binary    = (n_outputs == 1)

    inputs = Input(shape=(input_dim,))
    h   = Dense(ORIG_H1, activation='relu', kernel_initializer='glorot_uniform')(inputs)
    h   = Dense(ORIG_H2, activation='relu', kernel_initializer='glorot_uniform')(h)
    out = Dense(n_outputs,
                activation='sigmoid' if binary else 'softmax',
                kernel_initializer='glorot_uniform')(h)

    model = Model(inputs=inputs, outputs=out)
    model.compile(optimizer=Adam(learning_rate=ORIG_LR),
                  loss='binary_crossentropy' if binary else 'sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    key = (input_dim, n_outputs)
    _INITIAL_WEIGHTS.setdefault(key, model.get_weights())
    model.set_weights(_INITIAL_WEIGHTS[key])
    return model


def original_early_stopping():
    """The callback used by the original model."""
    return EarlyStopping(monitor='val_loss', mode='min', patience=ORIG_PATIENCE)


def _glorot(fan_in, fan_out, rng):
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return rng.uniform(-limit, limit, size=(fan_in, fan_out)).astype('float32')


class OriginalArchClassifier(BaseEstimator, ClassifierMixin):
    """Original Model architecture behind a scikit-learn interface:
    fit(X, y, sample_weight) and predict(X) returning 0/1.

    This is what ExponentiatedGradientReduction and GridSearchReduction need as
    their `estimator`, and it is also the downstream model in the sequential
    section. It inherits BaseEstimator/ClassifierMixin so that get_params,
    set_params, clone() and __sklearn_tags__ all work - the reductions clone
    the estimator once per iteration, and scikit-learn >= 1.6 reaches for
    __sklearn_tags__ on any object passed as an estimator. Only hyperparameters
    live in __init__; the Keras model is built in fit() so a cloned, unfitted
    copy stays cheap to copy.

    If eager execution is ever turned off (e.g. you add back a TF1 algorithm
    such as AdversarialDebiasing), Keras cannot train, so this class falls back
    to a graph-mode twin of the same network: 8 -> 6 -> 1 sigmoid,
    glorot-uniform init, Adam lr=0.01, 25 epochs, batch 64. Identical
    architecture, different plumbing.

    Note there is no clear_session() call anywhere: EGR keeps every predictor it
    fits and averages them at predict time, so clearing the Keras session
    mid-run would invalidate models fairlearn still needs.
    """

    def __init__(self, n_outputs=1):
        self.n_outputs = n_outputs          # hyperparameters only

    # ---------------------------------------------------------------- fit
    def fit(self, X, y, sample_weight=None, validation_data=None):
        Xf = np.asarray(X, dtype='float32')
        yf = np.asarray(y, dtype='float32').ravel()
        sw = (np.ones(len(Xf), dtype='float32') if sample_weight is None
              else np.asarray(sample_weight, dtype='float32').ravel())

        self.classes_       = np.array([0, 1])
        self.n_features_in_ = Xf.shape[1]
        self.sess_          = None
        self.model_         = None

        if _tf.executing_eagerly():
            self.model_ = build_original_arch(input_dim=Xf.shape[1],
                                              n_outputs=self.n_outputs)
            callbacks = [original_early_stopping()] if validation_data is not None else []
            self.model_.fit(Xf, yf, sample_weight=sw,
                            epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
                            validation_data=validation_data, callbacks=callbacks)
        else:
            self._fit_graph(Xf, yf.reshape(-1, 1), sw.reshape(-1, 1))
        return self

    def _fit_graph(self, Xf, yf, sw):
        """Graph-mode twin, used only when eager execution is off."""
        v1 = _tf.compat.v1

        n_in = Xf.shape[1]
        rng  = np.random.RandomState(42)
        w1, w2, w3 = (_glorot(n_in, ORIG_H1, rng),
                      _glorot(ORIG_H1, ORIG_H2, rng),
                      _glorot(ORIG_H2, 1, rng))

        graph = v1.Graph()
        with graph.as_default():
            xp = v1.placeholder(v1.float32, [None, n_in])
            yp = v1.placeholder(v1.float32, [None, 1])
            wp = v1.placeholder(v1.float32, [None, 1])

            W1, b1 = v1.Variable(w1), v1.Variable(np.zeros(ORIG_H1, dtype='float32'))
            W2, b2 = v1.Variable(w2), v1.Variable(np.zeros(ORIG_H2, dtype='float32'))
            W3, b3 = v1.Variable(w3), v1.Variable(np.zeros(1, dtype='float32'))

            h      = v1.nn.relu(v1.matmul(xp, W1) + b1)
            h      = v1.nn.relu(v1.matmul(h,  W2) + b2)
            logits = v1.matmul(h, W3) + b3
            prob   = v1.nn.sigmoid(logits)

            loss = v1.reduce_mean(
                wp * v1.nn.sigmoid_cross_entropy_with_logits(labels=yp, logits=logits))
            train_op = v1.train.AdamOptimizer(ORIG_LR).minimize(loss)
            init_op  = v1.global_variables_initializer()

        self.sess_ = v1.Session(graph=graph)
        self.sess_.run(init_op)
        self._xp, self._prob = xp, prob

        n, order = len(Xf), np.arange(len(Xf))
        for _ in range(ORIG_EPOCHS):
            rng.shuffle(order)
            for start in range(0, n, ORIG_BATCH_SIZE):
                idx = order[start:start + ORIG_BATCH_SIZE]
                self.sess_.run(train_op, feed_dict={xp: Xf[idx],
                                                    yp: yf[idx],
                                                    wp: sw[idx]})

    # ------------------------------------------------------------ predict
    def predict_proba(self, X):
        Xp = np.asarray(X, dtype='float32')
        if self.sess_ is not None:
            p = self.sess_.run(self._prob, feed_dict={self._xp: Xp})
        else:
            p = self.model_.predict(Xp, verbose=0)
        if self.n_outputs == 1:                    # match sklearn's (n, 2) layout
            p = np.asarray(p).reshape(-1, 1)
            return np.hstack([1.0 - p, p])
        return np.asarray(p)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


print("Features:", N_FEATURES, "| architecture used by every section below:")
build_original_arch().summary()


# Mitigation Algorithms

# Preprocessing Algorithms:

## 1- Reweighing

In [ ]:
from aif360.algorithms.preprocessing import Reweighing
from aif360.datasets import StandardDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tqdm import tqdm


# Architecture: the Original Model (n_features -> 8 -> 6 -> 4 softmax).
# build_original_arch() returns it compiled, with the shared initial weights.

y_pred_1 = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

groups = [
    {'Gender': 0, 'Ethnicity': 0.0},
    {'Gender': 0, 'Ethnicity': 1.0},
    {'Gender': 0, 'Ethnicity': 2.0},
    {'Gender': 0, 'Ethnicity': 4.0},
    {'Gender': 1, 'Ethnicity': 0.0},
    {'Gender': 1, 'Ethnicity': 1.0},
    {'Gender': 1, 'Ethnicity': 2.0},
    {'Gender': 1, 'Ethnicity': 4.0}
]

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    X_Train, X_Test = X.iloc[train_index], X.iloc[test_index]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    # Use correct label — Status, not Age
    train_df = pd.concat([X_Train, y_Train.rename('Status')], axis=1)
    weights  = np.ones(len(y_Train), dtype=np.float64)

    n_total = len(y_Train)

    for priv_group in groups:
        priv_mask = (
            (train_df['Gender']    == priv_group['Gender']) &
            (train_df['Ethnicity'] == priv_group['Ethnicity'])
        )
        unpriv_mask = ~priv_mask

        n_priv   = priv_mask.sum()
        n_unpriv = unpriv_mask.sum()

        if n_priv == 0 or n_unpriv == 0:
            continue

        for cls in y_Train.unique():
            cls_mask     = (train_df['Status'] == cls)
            n_cls        = cls_mask.sum()
            n_priv_cls   = (priv_mask   & cls_mask).sum()
            n_unpriv_cls = (unpriv_mask & cls_mask).sum()

            if n_priv_cls == 0 or n_unpriv_cls == 0:
                continue

            w_priv   = (n_cls / n_total) * (n_priv   / n_total) / (n_priv_cls   / n_total)
            w_unpriv = (n_cls / n_total) * (n_unpriv / n_total) / (n_unpriv_cls / n_total)

            weights[ priv_mask  & cls_mask] *= w_priv
            weights[unpriv_mask & cls_mask] *= w_unpriv

    weights /= np.mean(weights)
    weights  = np.clip(weights, 0.1, 10.0)

    dense_model = build_original_arch(n_outputs=4)
    fit_es(dense_model, X_Train.values.astype('float32'),
        y_Train.values.astype('float32'),
        sample_weight=weights,
        epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
        callbacks=[original_early_stopping()]
    )
    preds = dense_model.predict(X_Test.values.astype('float32'), verbose=0).squeeze()
    y_pred_1.iloc[test_index] = preds
    scores.append(accuracy_score(y_Test, np.argmax(preds, axis=1)))

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores))

In [49]:
y_pred_1.to_csv('y_pred_Reweighing.csv',index=False)

In [50]:
y_pred_1 = pd.read_csv('y_pred_Reweighing.csv')
accuracy_score(y_Test.values.astype('float32'), np.argmax(preds, axis=1))

Out[39]: 0.530442035029191


In [51]:
# ── Fairness metrics for RW (appended to metrics_values) ──
results_rw = evaluate_fairness(y, y_pred_1, X, algorithm='RW')
results_rw


=== RW ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
       RW             Gender Overall (all classes)     0.533                0.011                            NaN                        NaN         2
       RW             Gender             Non-COVID     0.533                0.011                          0.011                      0.022         2
       RW             Gender                 Covid     0.533                0.011                          0.011                      0.022         2
       RW          Ethnicity Overall (all classes)     0.533                0.078                            NaN                        NaN         5
       RW          Ethnicity             Non-COVID     0.533                0.078                          0.207                      0.256         5
       RW          Ethnicity                 Covid     0.533                0.078        

#### Metrics

In [ ]:
# AIF360 metrics, Gender (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_1, 'Gender')

In [ ]:
# AIF360 metrics, Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_1, 'Ethnicity')

In [ ]:
# AIF360 metrics, Gender+Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_1, 'Gender+Ethnicity')

## 2- Disparate Impact Remover

In [ ]:
from aif360.algorithms.preprocessing import DisparateImpactRemover
from aif360.datasets import StandardDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tqdm import tqdm

# Architecture: the Original Model (n_features -> 8 -> 6 -> 4 softmax),
# built by build_original_arch() from the shared setup cell.

# CV and OOF Predictions
y_pred_2 = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

# Define the groups
groups = [
    {'Gender': 0, 'Ethnicity': 0.0},
    {'Gender': 0, 'Ethnicity': 1.0},
    {'Gender': 0, 'Ethnicity': 2.0},
    {'Gender': 1, 'Ethnicity': 0.0},
    {'Gender': 1, 'Ethnicity': 1.0},
    {'Gender': 1, 'Ethnicity': 2.0},
    {'Gender': 0, 'Ethnicity': 4.0},
    {'Gender': 1, 'Ethnicity': 4.0}
]

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    # Initialize a DataFrame to hold the transformed features
    transformed_X_Train = pd.DataFrame(index=X_Train.index, columns=X_Train.columns)

    for priv_group in tqdm(groups):
        # Determine unprivileged groups
        unprivileged_groups = [group for group in groups if group != priv_group]

        # Convert to AIF360 format
        aif360_train = StandardDataset(pd.concat([X_Train, y_Train], axis=1),
                                       label_name='Status', favorable_classes=[1],
                                       protected_attribute_names=['Gender', 'Ethnicity'],
                                       privileged_classes=[list(priv_group.values())])

        # Apply Disparate Impact Remover
        DIR = DisparateImpactRemover(repair_level=1.0)
        aif360_train = DIR.fit_transform(aif360_train)

        # Update the transformed features
        transformed_X_Train = transformed_X_Train.add(pd.DataFrame(aif360_train.features, columns=X.columns, index=X_Train.index), fill_value=0)

    # Average the transformed features
    transformed_X_Train /= len(groups)

    # Initialize the Original Model architecture (compiled, shared init weights)
    dense_model = build_original_arch(n_outputs=4)

    # Train the model with the transformed dataset
    fit_es(dense_model, transformed_X_Train.values.astype('float32'),
                    y_Train.values.astype('float32'),
                    epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
                    callbacks=[original_early_stopping()])

    # Make predictions and evaluate
    preds = dense_model.predict(X_Test.values.astype('float32'), verbose=0).squeeze()
    y_pred_2.iloc[test_index, :] = preds
    scores.append(accuracy_score(y_Test.values.astype('float32'), np.argmax(preds, axis=1)))

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [58]:
y_pred_2.to_csv('y_pred_DisparateImpactRemover.csv',index=False)

In [59]:
y_pred_2 = pd.read_csv('y_pred_DisparateImpactRemover.csv')
accuracy_score(y_Test.values.astype('float32'), np.argmax(preds, axis=1))

Out[46]: 0.5471226021684737


In [60]:
# ── Fairness metrics for DIR (appended to metrics_values) ──
results_dir = evaluate_fairness(y, y_pred_2, X, algorithm='DIR')
results_dir


=== DIR ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
      DIR             Gender Overall (all classes)     0.573                0.028                            NaN                        NaN         2
      DIR             Gender             Non-COVID     0.573                0.028                          0.025                      0.053         2
      DIR             Gender                 Covid     0.573                0.028                          0.025                      0.053         2
      DIR          Ethnicity Overall (all classes)     0.573                0.173                            NaN                        NaN         5
      DIR          Ethnicity             Non-COVID     0.573                0.173                          0.371                      0.375         5
      DIR          Ethnicity                 Covid     0.573                0.173       

### Metrics

In [ ]:
# AIF360 metrics, Gender (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_2, 'Gender')

In [ ]:
# AIF360 metrics, Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_2, 'Ethnicity')

In [ ]:
# AIF360 metrics, Gender+Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_2, 'Gender+Ethnicity')

## 3- Learning Fair Representations (LFR)

In [66]:
import aif360.algorithms.preprocessing.lfr_helpers as lfr_helpers
import numpy as np

# Patch: replace None bounds with finite values before optimization
original_fmin = __import__('scipy').optimize.fmin_l_bfgs_b

def patched_fmin(func, x0, *args, bounds=None, **kwargs):
    if bounds is not None:
        bounds = [(l if l is not None else -np.inf,
                   u if u is not None else np.inf) 
                  for l, u in bounds]
    return original_fmin(func, x0, *args, bounds=bounds, **kwargs)

import scipy.optimize
scipy.optimize.fmin_l_bfgs_b = patched_fmin

In [ ]:
from aif360.algorithms.preprocessing import LFR
from aif360.datasets import StandardDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tqdm import tqdm

# Architecture: the Original Model (n_features -> 8 -> 6 -> 4 softmax),
# built by build_original_arch() from the shared setup cell.

# CV and OOF Predictions
y_pred_3 = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index, dtype='float32')
scores = []

# Define the groups
groups = [
                {'Gender': 0, 'Ethnicity': 0.0},
                {'Gender': 0, 'Ethnicity': 1.0},
                {'Gender': 0, 'Ethnicity': 2.0},
                {'Gender': 1, 'Ethnicity': 0.0},
                {'Gender': 1, 'Ethnicity': 1.0},
                {'Gender': 1, 'Ethnicity': 2.0},
                {'Gender': 0, 'Ethnicity': 4.0},
                {'Gender': 1, 'Ethnicity': 4.0}
            ]

# Stratified K-Fold
for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    # Initialize a DataFrame to hold the transformed features
    transformed_X_Train = pd.DataFrame(index=X_Train.index, columns=X_Train.columns)

    for priv_group in tqdm(groups):
        # Determine unprivileged groups
        unprivileged_groups = [group for group in groups if group != priv_group]

        # Apply LFR for each pair of unprivileged and privileged groups
        for unpriv_group in unprivileged_groups:
            # Convert to AIF360 format
            aif360_train = StandardDataset(pd.concat([X_Train, y_Train], axis=1),
                                           label_name='Status', favorable_classes=[STATUS_MAPPING['Covid']],
                                           protected_attribute_names=['Gender', 'Ethnicity'],
                                           privileged_classes=[list(priv_group.values())])

            # Apply Learning Fair Representations
            lfr = LFR(unprivileged_groups=[unpriv_group], privileged_groups=[priv_group],seed=42,k=4,verbose=0)
            lfr = lfr.fit(aif360_train)
            aif360_train = lfr.transform(aif360_train)

            # Update the transformed features
            transformed_X_Train = transformed_X_Train.add(pd.DataFrame(aif360_train.features, columns=X.columns, index=X_Train.index), fill_value=0)

    # Average the transformed features
    transformed_X_Train /= len(groups) * (len(groups) - 1)

    # Initialize the Original Model architecture (compiled, shared init weights)
    dense_model = build_original_arch(n_outputs=4)

    # Train the model with the transformed dataset
    fit_es(dense_model, transformed_X_Train.values.astype('float32'),
                    y_Train.values.astype('float32'),
                    epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
                    callbacks=[original_early_stopping()])

    # Make predictions and evaluate
    preds = dense_model.predict(X_Test.values.astype('float32'), verbose=0).squeeze()
    y_pred_3.iloc[test_index, :] = preds
    scores.append(accuracy_score(y_Test.values.astype('float32'), np.argmax(preds, axis=1)))

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [68]:
y_pred_3.to_csv('y_pred_LFR.csv',index=False)

In [69]:
# ── Fairness metrics for LFR (appended to metrics_values) ──
results_lfr = evaluate_fairness(y, y_pred_3, X, algorithm='LFR')
results_lfr


=== LFR ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
      LFR             Gender Overall (all classes)     0.518                0.014                            NaN                        NaN         2
      LFR             Gender             Non-COVID     0.518                0.014                          0.033                      0.047         2
      LFR             Gender                 Covid     0.518                0.014                          0.033                      0.047         2
      LFR          Ethnicity Overall (all classes)     0.518                0.113                            NaN                        NaN         5
      LFR          Ethnicity             Non-COVID     0.518                0.113                          0.286                      0.291         5
      LFR          Ethnicity                 Covid     0.518                0.113       

### Metrics

In [ ]:
# AIF360 metrics, Gender (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_3, 'Gender')

In [ ]:
# AIF360 metrics, Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_3, 'Ethnicity')

In [ ]:
# AIF360 metrics, Gender+Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_3, 'Gender+Ethnicity')

# In-processing Algorithms:

### 1- Exponentiated Gradient Reduction

In [ ]:
# ===========================================================================
# In-processing: Exponentiated Gradient Reduction
#
# EGR takes an `estimator` implementing fit(X, y, sample_weight) and predict(X)
# with 0/1 output, which is exactly what OriginalArchClassifier provides - so
# the model being debiased is the Original Model architecture
# (n_features -> 8 -> 6 -> 1 sigmoid, Adam lr=0.01, 25 epochs, batch 64).
#
# EGR is binary-only, so the one-vs-rest loop over the classes is kept.
# Unlike AdversarialDebiasing this is pure Python/fairlearn: no TF1 session,
# no disable_v2_behavior(), so the Keras path runs normally.
# ===========================================================================
from aif360.algorithms.inprocessing import ExponentiatedGradientReduction
from aif360.datasets import StandardDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from tqdm import tqdm

# The reductions sit behind an optional dependency. Fail loudly here rather
# than with a confusing NameError from inside aif360.
try:
    import fairlearn.reductions as _red   # noqa: F401
except ImportError:
    raise ImportError("fairlearn is required. Run: pip install 'aif360[Reductions]'")


def make_egr():
    """EGR with a version-tolerant iteration-budget keyword
    (max_iter in aif360 >= 0.5, T in 0.4)."""
    kwargs = dict(estimator=OriginalArchClassifier(n_outputs=1),  # fresh, unfitted
                  constraints='EqualizedOdds',                    # or 'DemographicParity'
                  drop_prot_attr=False)   # keep all features -> same n_features as original
    try:
        return ExponentiatedGradientReduction(max_iter=EGR_MAX_ITER, **kwargs)
    except TypeError:
        return ExponentiatedGradientReduction(T=EGR_MAX_ITER, **kwargs)


# CV and OOF Predictions
y_pred_4 = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

# Define the groups
groups = [
    {'Gender': 0},
    {'Gender': 1},
]
priv_group = groups[0]
unprivileged_groups = [groups[1]]

# Unique values of the target
unique_targets = np.sort(y.unique())

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    for target_value in unique_targets:
        # Convert the target to binary for the current class
        y_Train_binary = (y_Train == target_value).astype(int).rename('Status')
        y_Test_binary  = (y_Test  == target_value).astype(int).rename('Status')

        # Convert to AIF360 format
        aif360_train = StandardDataset(pd.concat([X_Train, y_Train_binary], axis=1),
                                       label_name='Status', favorable_classes=[1],
                                       protected_attribute_names=['Gender'],
                                       privileged_classes=[[priv_group['Gender']]])

        aif360_test = StandardDataset(pd.concat([X_Test, y_Test_binary], axis=1),
                                      label_name='Status', favorable_classes=[1],
                                      protected_attribute_names=['Gender'],
                                      privileged_classes=[[priv_group['Gender']]])

        # Initialize and train Exponentiated Gradient Reduction
        egr = make_egr()
        egr.fit(aif360_train)

        # Make predictions and evaluate
        preds = np.asarray(egr.predict(aif360_test).labels).ravel()
        y_pred_4.iloc[test_index, int(target_value)] = preds
        scores.append(accuracy_score(y_Test_binary.values.astype(int),
                                     preds.astype(int)))

print("Overall Accuracy:", accuracy_score(y, np.argmax(y_pred_4.values, axis=1)))
print("Mean (per-class, one-vs-rest):", np.mean(scores), "\nSTD: ", np.std(scores), '\n')


In [77]:
y_pred_4.to_csv('y_pred_EGR.csv',index=False)

In [78]:
# ── Fairness metrics for EGR (appended to metrics_values) ──
results_egr = evaluate_fairness(y, y_pred_4, X, algorithm='EGR')
results_egr


=== EGR ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
      EGR             Gender Overall (all classes)     0.624                0.002                            NaN                        NaN         2
      EGR             Gender             Non-COVID     0.624                0.002                          0.000                      0.002         2
      EGR             Gender                 Covid     0.624                0.002                          0.000                      0.002         2
      EGR          Ethnicity Overall (all classes)     0.624                0.105                            NaN                        NaN         5
      EGR          Ethnicity             Non-COVID     0.624                0.105                          0.797                      0.817         5
      EGR          Ethnicity                 Covid     0.624                0.105       

### Metrics

In [ ]:
# AIF360 metrics, Gender (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_4, 'Gender')

In [ ]:
# AIF360 metrics, Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_4, 'Ethnicity')

In [ ]:
# AIF360 metrics, Gender+Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_4, 'Gender+Ethnicity')

### 2- Grid Search Reduction

In [ ]:
# ===========================================================================
# In-processing: Grid Search Reduction
#
# Same estimator contract as EGR, so the model being debiased is again the
# Original Model architecture (n_features -> 8 -> 6 -> 1 sigmoid). The
# difference is the search: EGR runs exponentiated gradient and returns a
# randomized classifier, GSR sweeps a grid of Lagrange multipliers and returns
# the single deterministic classifier with the lowest error under constraint.
# ===========================================================================
from aif360.algorithms.inprocessing import GridSearchReduction
from aif360.datasets import StandardDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from tqdm import tqdm

try:
    import fairlearn.reductions as _red   # noqa: F401
except ImportError:
    raise ImportError("fairlearn is required. Run: pip install 'aif360[Reductions]'")


def make_gsr():
    """GSR on the Original Model architecture."""
    return GridSearchReduction(
        estimator=OriginalArchClassifier(n_outputs=1),   # fresh, unfitted
        constraints='EqualizedOdds',                     # or 'DemographicParity'
        prot_attr='Gender',
        grid_size=GSR_GRID_SIZE,
        drop_prot_attr=False)   # keep all features -> same n_features as original


# CV and OOF Predictions
y_pred_5 = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

# Define the groups
groups = [
    {'Gender': 0},
    {'Gender': 1},
]
priv_group = groups[0]
unprivileged_groups = [groups[1]]

# Unique values of the target
unique_targets = np.sort(y.unique())

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    for target_value in unique_targets:
        # Convert the target to binary for the current class
        y_Train_binary = (y_Train == target_value).astype(int).rename('Status')
        y_Test_binary  = (y_Test  == target_value).astype(int).rename('Status')

        # Convert to AIF360 format
        aif360_train = StandardDataset(pd.concat([X_Train, y_Train_binary], axis=1),
                                       label_name='Status', favorable_classes=[1],
                                       protected_attribute_names=['Gender'],
                                       privileged_classes=[[priv_group['Gender']]])

        aif360_test = StandardDataset(pd.concat([X_Test, y_Test_binary], axis=1),
                                      label_name='Status', favorable_classes=[1],
                                      protected_attribute_names=['Gender'],
                                      privileged_classes=[[priv_group['Gender']]])

        # Initialize and train Grid Search Reduction
        gsr = make_gsr()
        gsr.fit(aif360_train)

        # Make predictions and evaluate
        preds = np.asarray(gsr.predict(aif360_test).labels).ravel()
        y_pred_5.iloc[test_index, int(target_value)] = preds
        scores.append(accuracy_score(y_Test_binary.values.astype(int),
                                     preds.astype(int)))

print("Overall Accuracy:", accuracy_score(y, np.argmax(y_pred_5.values, axis=1)))
print("Mean (per-class, one-vs-rest):", np.mean(scores), "\nSTD: ", np.std(scores), '\n')


In [85]:
y_pred_5.to_csv('y_pred_GSR.csv',index=False)

In [86]:
# ── Fairness metrics for GSR (appended to metrics_values) ──
results_gsr = evaluate_fairness(y, y_pred_5, X, algorithm='GSR')
results_gsr


=== GSR ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
      GSR             Gender Overall (all classes)     0.624                0.010                            NaN                        NaN         2
      GSR             Gender             Non-COVID     0.624                0.010                          0.007                      0.017         2
      GSR             Gender                 Covid     0.624                0.010                          0.007                      0.017         2
      GSR          Ethnicity Overall (all classes)     0.624                0.129                            NaN                        NaN         5
      GSR          Ethnicity             Non-COVID     0.624                0.129                          0.801                      0.808         5
      GSR          Ethnicity                 Covid     0.624                0.129       

### Metrics

In [ ]:
# AIF360 metrics, Gender (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_5, 'Gender')

In [ ]:
# AIF360 metrics, Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_5, 'Ethnicity')

In [ ]:
# AIF360 metrics, Gender+Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_5, 'Gender+Ethnicity')

# Postprocessing Algorithms:

### 1- Equalized Odds Postprocessing

In [ ]:
from aif360.algorithms.postprocessing import EqOddsPostprocessing, CalibratedEqOddsPostprocessing
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# Version 3: binary task -> one model for the Covid class. The base model is trained
# on part of the training fold, the post-processor is fitted on the held-out
# calibration part of the training fold (V2 fitted it on the test fold itself), and
# the fitted post-processor is then applied to the test fold.
privileged_groups = [{'Gender': 1}]
unprivileged_groups = [{'Gender': 0}]

n_classes = N_CLASSES
covid = STATUS_MAPPING['Covid']
y_pred_all_classes = np.zeros((len(y), n_classes))

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    fit_index, cal_index = calibration_split(train_index)
    X_Fit, X_Cal, X_Test = X.iloc[fit_index], X.iloc[cal_index], X.iloc[test_index]
    y_Fit, y_Cal = (y.iloc[fit_index] == covid).astype(int), (y.iloc[cal_index] == covid).astype(int)

    dense_model = build_original_arch(input_dim=X.shape[1], n_outputs=1)
    dense_model.fit(X_Fit.values.astype('float32'), y_Fit.values.astype('float32'),
                    epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
                    validation_data=(X_Cal.values.astype('float32'), y_Cal.values.astype('float32')),
                    callbacks=[original_early_stopping()])
    p_cal  = dense_model.predict(X_Cal.values.astype('float32'),  verbose=0).ravel()
    p_test = dense_model.predict(X_Test.values.astype('float32'), verbose=0).ravel()

    cal_true = gender_dataset(X_Cal, y_Cal.values)
    cal_pred = gender_dataset(X_Cal, p_cal >= 0.5, p_cal)
    test_pred = gender_dataset(X_Test, p_test >= 0.5, p_test)

    eq_odds = CalibratedEqOddsPostprocessing(unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups, seed=SEED)
    eq_odds = eq_odds.fit(cal_true, cal_pred)
    out = eq_odds.predict(test_pred).labels.ravel()
    y_pred_all_classes[test_index, covid] = out
    y_pred_all_classes[test_index, 1 - covid] = 1 - out

y_pred_final = y_pred_all_classes[:, covid].astype(int)
overall_accuracy = accuracy_score(y, y_pred_final)
print("Overall Accuracy:", overall_accuracy)

In [94]:
pd.DataFrame(y_pred_final).to_csv('y_pred_EOP.csv',index=False)

In [95]:
# ── Fairness metrics for EOP (appended to metrics_values) ──
# y_pred_final is overwritten by the next block, so evaluate it here.
results_eop = evaluate_fairness(y, y_pred_final, X, algorithm='EOP')
results_eop


=== EOP ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
      EOP             Gender Overall (all classes)       0.5                0.001                            NaN                        NaN         2
      EOP             Gender             Non-COVID       0.5                0.001                            0.0                        0.0         2
      EOP             Gender                 Covid       0.5                0.001                            0.0                        0.0         2
      EOP          Ethnicity Overall (all classes)       0.5                0.306                            NaN                        NaN         5
      EOP          Ethnicity             Non-COVID       0.5                0.306                            0.0                        0.0         5
      EOP          Ethnicity                 Covid       0.5                0.306       

In [ ]:
# AIF360 metrics, Gender (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_final, 'Gender')

In [ ]:
# AIF360 metrics, Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_final, 'Ethnicity')

In [ ]:
# AIF360 metrics, Gender+Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_final, 'Gender+Ethnicity')

### 2- Reject Option Classification

In [ ]:
from aif360.algorithms.postprocessing import RejectOptionClassification
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# Version 3: same protocol as the EOP cell above (fit on a calibration split of the
# training fold, apply to the test fold); probabilities are passed as scores.
privileged_groups = [{'Gender': 1}]
unprivileged_groups = [{'Gender': 0}]

n_classes = N_CLASSES
covid = STATUS_MAPPING['Covid']
y_pred_all_classes = np.zeros((len(y), n_classes))

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    fit_index, cal_index = calibration_split(train_index)
    X_Fit, X_Cal, X_Test = X.iloc[fit_index], X.iloc[cal_index], X.iloc[test_index]
    y_Fit, y_Cal = (y.iloc[fit_index] == covid).astype(int), (y.iloc[cal_index] == covid).astype(int)

    dense_model = build_original_arch(input_dim=X.shape[1], n_outputs=1)
    dense_model.fit(X_Fit.values.astype('float32'), y_Fit.values.astype('float32'),
                    epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
                    validation_data=(X_Cal.values.astype('float32'), y_Cal.values.astype('float32')),
                    callbacks=[original_early_stopping()])
    p_cal  = dense_model.predict(X_Cal.values.astype('float32'),  verbose=0).ravel()
    p_test = dense_model.predict(X_Test.values.astype('float32'), verbose=0).ravel()

    cal_true = gender_dataset(X_Cal, y_Cal.values)
    cal_pred = gender_dataset(X_Cal, p_cal >= 0.5, p_cal)
    test_pred = gender_dataset(X_Test, p_test >= 0.5, p_test)

    roc = RejectOptionClassification(unprivileged_groups=unprivileged_groups,
                                     privileged_groups=privileged_groups,
                                     low_class_thresh=0.01, high_class_thresh=0.99,
                                     num_class_thresh=100, num_ROC_margin=50,
                                     metric_name="Statistical parity difference",
                                     metric_ub=0.05, metric_lb=-0.05)
    roc = roc.fit(cal_true, cal_pred)
    out = roc.predict(test_pred).labels.ravel()
    y_pred_all_classes[test_index, covid] = out
    y_pred_all_classes[test_index, 1 - covid] = 1 - out

y_pred_final = y_pred_all_classes[:, covid].astype(int)
overall_accuracy = accuracy_score(y, y_pred_final)
print("Overall Accuracy:", overall_accuracy)

In [101]:
pd.DataFrame(y_pred_final).to_csv('y_pred_ROC.csv',index=False)

In [102]:
# ── Fairness metrics for ROC (appended to metrics_values) ──
# y_pred_final is overwritten by the next block, so evaluate it here.
results_roc = evaluate_fairness(y, y_pred_final, X, algorithm='ROC')
results_roc


=== ROC ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
      ROC             Gender Overall (all classes)       0.5                0.001                            NaN                        NaN         2
      ROC             Gender             Non-COVID       0.5                0.001                            0.0                        0.0         2
      ROC             Gender                 Covid       0.5                0.001                            0.0                        0.0         2
      ROC          Ethnicity Overall (all classes)       0.5                0.306                            NaN                        NaN         5
      ROC          Ethnicity             Non-COVID       0.5                0.306                            0.0                        0.0         5
      ROC          Ethnicity                 Covid       0.5                0.306       

In [ ]:
# AIF360 metrics, Gender (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_final, 'Gender')

In [ ]:
# AIF360 metrics, Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_final, 'Ethnicity')

In [ ]:
# AIF360 metrics, Gender+Ethnicity (computed with the shared helper `aif_report`)
_ = aif_report(y_pred_final, 'Gender+Ethnicity')

# Sequential Models

## Common setup for the sequential models

In [108]:
import pandas as pd
import numpy as np

from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing, DisparateImpactRemover
from aif360.algorithms.inprocessing import (ExponentiatedGradientReduction,
                                            GridSearchReduction)
from aif360.algorithms.postprocessing import CalibratedEqOddsPostprocessing

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# NOTE: this cell used to call tf.compat.v1.disable_eager_execution() because
# AdversarialDebiasing needs a TF1 graph and session. The two reductions are
# pure Python/fairlearn, so eager execution stays on and the downstream network
# trains through the normal Keras path. The tf.reset_default_graph() /
# tf.Session() calls in the blocks below are gone for the same reason.

# ---------------------------------------------------------------------------
# Downstream model for all four sequential blocks, and the estimator handed to
# the two reductions: the Original Model architecture (n_features -> 8 -> 6)
# with a sigmoid head, wrapped in the sklearn-style OriginalArchClassifier so
# the blocks keep their .fit(...) / .predict_proba(...) calls.
# ---------------------------------------------------------------------------
def create_seq_model():
    return OriginalArchClassifier(n_outputs=1)


# Protected attribute / groups, same convention as the in-processing section
groups              = [{'Gender': 0}, {'Gender': 1}]
priv_group          = groups[0]
unprivileged_groups = [groups[1]]

# Groups for the postprocessing step, same convention as the Postprocessing section
ceop_privileged_groups   = [{'Gender': 1}]
ceop_unprivileged_groups = [{'Gender': 0}]

unique_targets = np.sort(y.unique())
n_classes      = len(unique_targets)
n_features     = X.shape[1]

print("classes:", unique_targets, "| features:", n_features)


classes: [0 1] | features: 11


### Bias metric helper

The three metric blocks below (Gender, Ethnicity, Gender+Ethnicity) are the same loops already used
throughout this notebook, wrapped in one function so each sequential model gets a single metrics cell
instead of three near-identical copies. The `safe_metric` guard is the one from the `Metrics` section
and keeps a zero-denominator group from killing the whole loop.

In [110]:
def safe_metric(fn):
    try:
        val = fn()
        return val if np.isfinite(val) else 0.0
    except (TypeError, ZeroDivisionError, FloatingPointError):
        return 0.0


def _run_metric_loop(pred_labels, protected_attribute_names, priv_class_lists, groups):
    """Shared inner loop: returns the max of each metric over target class x privileged class x group."""
    max_spd = max_aod = max_eod = max_theil = max_gei = max_di = -np.inf

    for age_cls in range(N_CLASSES):
        for priv_classes in priv_class_lists:
            aif360_dataset = StandardDataset(
                df=pd.concat([X, y.rename('Status')], axis=1),
                label_name='Status',
                favorable_classes=[age_cls],
                protected_attribute_names=protected_attribute_names,
                privileged_classes=priv_classes
            )
            aif360_dataset.labels = aif360_dataset.labels.astype(int)

            aif360_pred_dataset = aif360_dataset.copy()
            aif360_pred_dataset.labels = pred_labels

            for i, privileged_group in enumerate(groups):
                unprivileged_groups = groups[:i] + groups[i + 1:]

                metric = ClassificationMetric(
                    aif360_dataset, aif360_pred_dataset,
                    unprivileged_groups=unprivileged_groups,
                    privileged_groups=[privileged_group]
                )

                max_spd   = max(max_spd,   safe_metric(metric.statistical_parity_difference))
                max_aod   = max(max_aod,   safe_metric(metric.average_odds_difference))
                max_eod   = max(max_eod,   safe_metric(metric.equal_opportunity_difference))
                max_theil = max(max_theil, safe_metric(metric.theil_index))
                max_gei   = max(max_gei,   safe_metric(metric.generalized_entropy_index))
                max_di    = max(max_di,    safe_metric(metric.disparate_impact))

    return max_spd, max_aod, max_eod, max_theil, max_gei, max_di


def _print_block(header, vals):
    max_spd, max_aod, max_eod, max_theil, max_gei, max_di = vals
    print(header)
    print("Max Statistical Parity Difference (Ideal value = 0):", max_spd)
    print("Max Average Odds Difference (Ideal value = 0):", max_aod)
    print("Max Equal Opportunity Difference (Ideal value = 0):", max_eod)
    print("Max Theil Index (Ideal value = 0):", max_theil)
    print("Max Generalized Entropy Index (Ideal value = 0):", max_gei)
    print("Max Disparate Impact (Ideal value = 1):", max_di)
    print()


def bias_report(pred_labels, title):
    """Gender / Ethnicity / Gender+Ethnicity bias report for a vector of predicted class labels."""
    pred_labels = np.asarray(pred_labels).reshape(-1, 1).astype(int)

    print("=" * 70)
    print(title)
    print("=" * 70)

    # ---------------- Gender ----------------
    _print_block("Gender Bias Identification:", _run_metric_loop(
        pred_labels,
        protected_attribute_names=['Gender'],
        priv_class_lists=[[[0]], [[1]]],
        groups=[{'Gender': 0}, {'Gender': 1}]
    ))

    # ---------------- Ethnicity ----------------
    _print_block("Ethnicity Bias Identification:", _run_metric_loop(
        pred_labels,
        protected_attribute_names=['Ethnicity'],
        priv_class_lists=[[[0.0]], [[1.0]], [[2.0]], [[4.0]]],
        groups=[{'Ethnicity': 0.0}, {'Ethnicity': 1.0},
                {'Ethnicity': 2.0}, {'Ethnicity': 4.0}]
    ))

    # ---------------- Gender + Ethnicity ----------------
    _print_block("Gender+Ethnicity Bias Identification:", _run_metric_loop(
        pred_labels,
        protected_attribute_names=['Gender', 'Ethnicity'],
        priv_class_lists=[[[g], [e]] for g in [0, 1] for e in [0.0, 1.0, 2.0, 4.0]],
        groups=[{'Gender': g, 'Ethnicity': e} for g in [0, 1] for e in [0.0, 1.0, 2.0, 4.0]]
    ))

## RW + EGR

In [112]:
# create_seq_model() now lives in the "Common setup" cell above and returns the
# Original Model architecture (n_features -> 8 -> 6 -> 1 sigmoid).
#
# This cell used to override it with a gradient-boosted tree ensemble, which is
# why the sequential blocks were not comparable to the Original Model. The
# override is kept here, commented out, in case you want it back.
#
# from sklearn.ensemble import HistGradientBoostingClassifier
#
# def create_seq_model():
#     return HistGradientBoostingClassifier(random_state=42)


In [ ]:
# ===========================================================================
# Sequential: Reweighing -> Exponentiated Gradient Reduction -> downstream
# ===========================================================================
y_pred_rw_egr = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    for target_value in unique_targets:
        y_Train_binary = (y_Train == target_value).astype(int).rename('Status')
        y_Test_binary  = (y_Test  == target_value).astype(int).rename('Status')

        aif360_train = StandardDataset(
            pd.concat([X_Train, y_Train_binary], axis=1),
            label_name='Status', favorable_classes=[1],
            protected_attribute_names=['Gender'],
            privileged_classes=[[priv_group['Gender']]]
        )

        # ---------------- Step 1: Reweighing ----------------
        RW = Reweighing(unprivileged_groups=unprivileged_groups,
                        privileged_groups=[priv_group])
        train_dataset_reweighted = RW.fit_transform(aif360_train)

        weights = np.asarray(train_dataset_reweighted.instance_weights, dtype=np.float64)
        weights = weights / np.mean(weights)
        weights = np.clip(weights, 0.1, 10.0)

        # ---------------- Step 2: EGR on the reweighted data ----------------
        # The reductions ignore dataset.instance_weights (they generate their
        # own cost-sensitive weights per iteration), so Reweighing's weights are
        # carried into the downstream model at Step 4 instead - which is what
        # the Adversarial Debiasing version of this block did as well.
        egr = ExponentiatedGradientReduction(
            estimator=create_seq_model(),        # original architecture
            constraints='EqualizedOdds',
            max_iter=EGR_MAX_ITER,
            drop_prot_attr=False)
        egr.fit(train_dataset_reweighted)

        # ---------------- Step 3: relabel the training data ----------------
        X_Train_transformed = train_dataset_reweighted.features
        y_Train_transformed = egr.predict(train_dataset_reweighted).labels.ravel()

        # ---------------- Step 4: downstream classifier ----------------
        # A hard one-vs-rest split can collapse to a single class; fall back to
        # the reweighed labels when that happens.
        if len(np.unique(y_Train_transformed)) < 2:
            y_Train_transformed = train_dataset_reweighted.labels.ravel()

        model = create_seq_model()
        model.fit(X_Train_transformed.astype('float32'),
                  y_Train_transformed.astype('int32'),
                  sample_weight=weights)

        preds = model.predict_proba(X_Test.values.astype('float32'))[:, 1]

        y_pred_rw_egr.iloc[test_index, int(target_value)] = preds
        scores.append(accuracy_score(y_Test_binary.values, (preds > 0.5).astype(int)))

y_pred_rw_egr_labels = np.argmax(y_pred_rw_egr.values, axis=1)

print("Overall Accuracy:", accuracy_score(y, y_pred_rw_egr_labels))
print("Mean (per-class, one-vs-rest):", np.mean(scores), "\nSTD: ", np.std(scores), '\n')


In [114]:
y_pred_rw_egr.to_csv('y_pred_Sequential_RW_EGR.csv', index=False)

In [115]:
bias_report(y_pred_rw_egr_labels, "Sequential: RW + EGR")

Sequential: RW + EGR
Gender Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.0005540638025007816
Max Average Odds Difference (Ideal value = 0): 0.000760745864743867
Max Equal Opportunity Difference (Ideal value = 0): 0.0059387101958394695
Max Theil Index (Ideal value = 0): 0.27207410419842704
Max Generalized Entropy Index (Ideal value = 0): 0.19553572724185886
Max Disparate Impact (Ideal value = 1): 1.0011588262535964

Ethnicity Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.8101536080534057
Max Average Odds Difference (Ideal value = 0): 0.7946488202637486
Max Equal Opportunity Difference (Ideal value = 0): 0.8061356216844021
Max Theil Index (Ideal value = 0): 0.27207410419842704
Max Generalized Entropy Index (Ideal value = 0): 0.19553572724185886
Max Disparate Impact (Ideal value = 1): 10.715336038744859

Gender+Ethnicity Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.5522273311743544
Max Average Odds

In [116]:
# ── Fairness metrics for RW+EGR (appended to metrics_values) ──
results_rw_egr = evaluate_fairness(y, y_pred_rw_egr, X, algorithm='RW+EGR')
results_rw_egr


=== RW+EGR ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
   RW+EGR             Gender Overall (all classes)     0.625                0.005                            NaN                        NaN         2
   RW+EGR             Gender             Non-COVID     0.625                0.005                          0.001                      0.006         2
   RW+EGR             Gender                 Covid     0.625                0.005                          0.001                      0.006         2
   RW+EGR          Ethnicity Overall (all classes)     0.625                0.118                            NaN                        NaN         5
   RW+EGR          Ethnicity             Non-COVID     0.625                0.118                          0.818                      0.832         5
   RW+EGR          Ethnicity                 Covid     0.625                0.118    

### DIR + GSR

In [ ]:
# ===========================================================================
# Sequential: Disparate Impact Remover -> Grid Search Reduction -> downstream
# ===========================================================================
y_pred_dir_gsr = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    for target_value in unique_targets:
        y_Train_binary = (y_Train == target_value).astype(int).rename('Status')
        y_Test_binary  = (y_Test  == target_value).astype(int).rename('Status')

        aif360_train = StandardDataset(
            pd.concat([X_Train, y_Train_binary], axis=1),
            label_name='Status', favorable_classes=[1],
            protected_attribute_names=['Gender'],
            privileged_classes=[[priv_group['Gender']]]
        )

        # ---------------- Step 1: Disparate Impact Remover ----------------
        DIR = DisparateImpactRemover(repair_level=1.0, sensitive_attribute='Gender')
        aif360_train = DIR.fit_transform(aif360_train)

        # ---------------- Step 2: Grid Search Reduction on the repaired data ----------------
        gsr = GridSearchReduction(
            estimator=create_seq_model(),        # original architecture
            constraints='EqualizedOdds',
            prot_attr='Gender',
            grid_size=GSR_GRID_SIZE,
            drop_prot_attr=False)
        gsr.fit(aif360_train)

        # ---------------- Step 3: relabel the training data ----------------
        X_Train_transformed = aif360_train.features
        y_Train_transformed = gsr.predict(aif360_train).labels.ravel()

        # ---------------- Step 4: downstream classifier ----------------
        if len(np.unique(y_Train_transformed)) < 2:
            y_Train_transformed = aif360_train.labels.ravel()

        model = create_seq_model()
        model.fit(X_Train_transformed.astype('float32'),
                  y_Train_transformed.astype('int32'))

        # As in the structured notebook, the test split is scored on raw
        # (unrepaired) features. Repair X_Test through DIR first if you want
        # train/test consistency.
        preds = model.predict_proba(X_Test.values.astype('float32'))[:, 1]

        y_pred_dir_gsr.iloc[test_index, int(target_value)] = preds
        scores.append(accuracy_score(y_Test_binary.values, (preds > 0.5).astype(int)))

y_pred_dir_gsr_labels = np.argmax(y_pred_dir_gsr.values, axis=1)

print("Overall Accuracy:", accuracy_score(y, y_pred_dir_gsr_labels))
print("Mean (per-class, one-vs-rest):", np.mean(scores), "\nSTD: ", np.std(scores), '\n')


In [119]:
y_pred_dir_gsr.to_csv('y_pred_Sequential_DIR_GSR.csv', index=False)

In [120]:
bias_report(y_pred_dir_gsr_labels, "Sequential: DIR + GSR")

Sequential: DIR + GSR
Gender Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.015414150724947506
Max Average Odds Difference (Ideal value = 0): 0.01561450529509123
Max Equal Opportunity Difference (Ideal value = 0): 0.0242053950909501
Max Theil Index (Ideal value = 0): 0.26394842253858236
Max Generalized Entropy Index (Ideal value = 0): 0.19035122533103305
Max Disparate Impact (Ideal value = 1): 1.031381681534668

Ethnicity Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.7338305927954655
Max Average Odds Difference (Ideal value = 0): 0.716791294159867
Max Equal Opportunity Difference (Ideal value = 0): 0.7229886681106193
Max Theil Index (Ideal value = 0): 0.26394842253858236
Max Generalized Entropy Index (Ideal value = 0): 0.19035122533103305
Max Disparate Impact (Ideal value = 1): 6.095856162993628

Gender+Ethnicity Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.4944041659387115
Max Average Odds Differ

In [121]:
# ── Fairness metrics for DIR+GSR (appended to metrics_values) ──
results_dir_gsr = evaluate_fairness(y, y_pred_dir_gsr, X, algorithm='DIR+GSR')
results_dir_gsr


=== DIR+GSR ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
  DIR+GSR             Gender Overall (all classes)      0.62                0.009                            NaN                        NaN         2
  DIR+GSR             Gender             Non-COVID      0.62                0.009                          0.015                      0.024         2
  DIR+GSR             Gender                 Covid      0.62                0.009                          0.015                      0.024         2
  DIR+GSR          Ethnicity Overall (all classes)      0.62                0.132                            NaN                        NaN         5
  DIR+GSR          Ethnicity             Non-COVID      0.62                0.132                          0.740                      0.729         5
  DIR+GSR          Ethnicity                 Covid      0.62                0.132   

### EGR + CEOP

In [ ]:
# ===========================================================================
# Sequential: Exponentiated Gradient Reduction -> downstream -> Calibrated Eq. Odds
# Version 3: stages 1-2 are fitted on part of the training fold; Calibrated
# Equalized Odds is fitted on the held-out calibration part of the training fold
# (V2 fitted it on the test fold) and then applied to the test fold.
# ===========================================================================
y_pred_all_classes_egr_ceop = np.zeros((len(y), n_classes))
scores = []

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    fit_index, cal_index = calibration_split(train_index)
    X_Train, X_Cal, X_Test = X.iloc[fit_index, :], X.iloc[cal_index, :], X.iloc[test_index, :]
    y_Train, y_Cal, y_Test = y.iloc[fit_index], y.iloc[cal_index], y.iloc[test_index]

    for target_value in unique_targets:
        y_Train_binary = (y_Train == target_value).astype(int).rename('Status')
        y_Cal_binary   = (y_Cal   == target_value).astype(int).rename('Status')
        y_Test_binary  = (y_Test  == target_value).astype(int).rename('Status')

        aif360_train = StandardDataset(
            pd.concat([X_Train, y_Train_binary], axis=1),
            label_name='Status', favorable_classes=[1],
            protected_attribute_names=['Gender'],
            privileged_classes=[[priv_group['Gender']]]
        )

        # ---------------- Step 1: Exponentiated Gradient Reduction ----------------
        egr = ExponentiatedGradientReduction(
            estimator=create_seq_model(),        # original architecture
            constraints='EqualizedOdds',
            max_iter=EGR_MAX_ITER,
            drop_prot_attr=False)
        egr.fit(aif360_train)

        X_Train_transformed = aif360_train.features
        y_Train_transformed = egr.predict(aif360_train).labels.ravel()

        # ---------------- Step 2: downstream classifier ----------------
        if len(np.unique(y_Train_transformed)) < 2:
            y_Train_transformed = aif360_train.labels.ravel()

        model = create_seq_model()
        model.fit(X_Train_transformed.astype('float32'),
                  y_Train_transformed.astype('int32'))

        p_cal  = model.predict_proba(X_Cal.values.astype('float32'))[:, 1]
        p_test = model.predict_proba(X_Test.values.astype('float32'))[:, 1]

        # ---------------- Calibrated Equalized Odds Postprocessing ----------------
        eq_odds = CalibratedEqOddsPostprocessing(
            unprivileged_groups=ceop_unprivileged_groups,
            privileged_groups=ceop_privileged_groups,
            seed=SEED
        )
        eq_odds = eq_odds.fit(gender_dataset(X_Cal, y_Cal_binary.values),
                              gender_dataset(X_Cal, p_cal >= 0.5, p_cal))
        transformed_preds = eq_odds.predict(gender_dataset(X_Test, p_test >= 0.5, p_test)).labels.ravel()

        y_pred_all_classes_egr_ceop[test_index, int(target_value)] = transformed_preds
        scores.append(accuracy_score(y_Test_binary.values, transformed_preds))

y_pred_egr_ceop_labels = np.argmax(y_pred_all_classes_egr_ceop, axis=1)

print("Overall Accuracy:", accuracy_score(y, y_pred_egr_ceop_labels))
print("Mean (per-class, one-vs-rest):", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [124]:
pd.DataFrame(y_pred_all_classes_egr_ceop).to_csv('y_pred_Sequential_EGR_CEOP.csv', index=False)

In [125]:
bias_report(y_pred_egr_ceop_labels, "Sequential: EGR + CEOP")

Sequential: EGR + CEOP
Gender Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.02632031277831215
Max Average Odds Difference (Ideal value = 0): 0.026508356768046165
Max Equal Opportunity Difference (Ideal value = 0): 0.03358057047461149
Max Theil Index (Ideal value = 0): 0.3028354551217354
Max Generalized Entropy Index (Ideal value = 0): 0.2168338803380111
Max Disparate Impact (Ideal value = 1): 1.0617284208481588

Ethnicity Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.7395343481915022
Max Average Odds Difference (Ideal value = 0): 0.7251442064407569
Max Equal Opportunity Difference (Ideal value = 0): 0.7402140816774962
Max Theil Index (Ideal value = 0): 0.3028354551217354
Max Generalized Entropy Index (Ideal value = 0): 0.2168338803380111
Max Disparate Impact (Ideal value = 1): 10.162861260165252

Gender+Ethnicity Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.4926162806221738
Max Average Odds Diffe

In [126]:
# ── Fairness metrics for EGR+CEOP (appended to metrics_values) ──
results_egr_ceop = evaluate_fairness(y, y_pred_all_classes_egr_ceop, X, algorithm='EGR+CEOP')
results_egr_ceop


=== EGR+CEOP ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
 EGR+CEOP             Gender Overall (all classes)     0.613                0.007                            NaN                        NaN         2
 EGR+CEOP             Gender             Non-COVID     0.613                0.007                          0.026                      0.034         2
 EGR+CEOP             Gender                 Covid     0.613                0.007                          0.026                      0.034         2
 EGR+CEOP          Ethnicity Overall (all classes)     0.613                0.111                            NaN                        NaN         5
 EGR+CEOP          Ethnicity             Non-COVID     0.613                0.111                          0.750                      0.778         5
 EGR+CEOP          Ethnicity                 Covid     0.613                0.111  

### RW + CEOP

This block has no in-processing stage, so there was no Adversarial Debiasing to
replace. It is unchanged apart from the downstream model, which is the Original
Model architecture like everything else.

In [ ]:
# ===========================================================================
# Sequential: Reweighing -> downstream -> Calibrated Eq. Odds Postprocessing
# Version 3: stages 1-2 are fitted on part of the training fold; Calibrated
# Equalized Odds is fitted on the held-out calibration part of the training fold
# (V2 fitted it on the test fold) and then applied to the test fold.
# ===========================================================================
y_pred_all_classes_rw_ceop = np.zeros((len(y), n_classes))
scores = []

for train_index, test_index in tqdm(grouped_cv(n_splits=5)):
    fit_index, cal_index = calibration_split(train_index)
    X_Train, X_Cal, X_Test = X.iloc[fit_index, :], X.iloc[cal_index, :], X.iloc[test_index, :]
    y_Train, y_Cal, y_Test = y.iloc[fit_index], y.iloc[cal_index], y.iloc[test_index]

    for target_value in unique_targets:
        y_Train_binary = (y_Train == target_value).astype(int).rename('Status')
        y_Cal_binary   = (y_Cal   == target_value).astype(int).rename('Status')
        y_Test_binary  = (y_Test  == target_value).astype(int).rename('Status')

        aif360_train = StandardDataset(
            pd.concat([X_Train, y_Train_binary], axis=1),
            label_name='Status', favorable_classes=[1],
            protected_attribute_names=['Gender'],
            privileged_classes=[[priv_group['Gender']]]
        )

        # ---------------- Step 1: Reweighing ----------------
        RW = Reweighing(unprivileged_groups=unprivileged_groups,
                        privileged_groups=[priv_group])
        train_dataset_reweighted = RW.fit_transform(aif360_train)

        weights = np.asarray(train_dataset_reweighted.instance_weights, dtype=np.float64)
        weights = weights / np.mean(weights)
        weights = np.clip(weights, 0.1, 10.0)

        X_Train_transformed = train_dataset_reweighted.features
        y_Train_transformed = train_dataset_reweighted.labels.ravel()

        # ---------------- Step 2: downstream classifier ----------------
        model = create_seq_model()
        model.fit(X_Train_transformed.astype('float32'),
                  y_Train_transformed.astype('int32'),
                  sample_weight=weights)

        p_cal  = model.predict_proba(X_Cal.values.astype('float32'))[:, 1]
        p_test = model.predict_proba(X_Test.values.astype('float32'))[:, 1]

        # ---------------- Calibrated Equalized Odds Postprocessing ----------------
        eq_odds = CalibratedEqOddsPostprocessing(
            unprivileged_groups=ceop_unprivileged_groups,
            privileged_groups=ceop_privileged_groups,
            seed=SEED
        )
        eq_odds = eq_odds.fit(gender_dataset(X_Cal, y_Cal_binary.values),
                              gender_dataset(X_Cal, p_cal >= 0.5, p_cal))
        transformed_preds = eq_odds.predict(gender_dataset(X_Test, p_test >= 0.5, p_test)).labels.ravel()

        y_pred_all_classes_rw_ceop[test_index, int(target_value)] = transformed_preds
        scores.append(accuracy_score(y_Test_binary.values, transformed_preds))

y_pred_rw_ceop_labels = np.argmax(y_pred_all_classes_rw_ceop, axis=1)

print("Overall Accuracy:", accuracy_score(y, y_pred_rw_ceop_labels))
print("Mean (per-class, one-vs-rest):", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [129]:
pd.DataFrame(y_pred_all_classes_rw_ceop).to_csv('y_pred_Sequential_RW_CEOP.csv', index=False)

In [130]:
bias_report(y_pred_rw_ceop_labels, "Sequential: RW + CEOP")

Sequential: RW + CEOP
Gender Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.03239864985216068
Max Average Odds Difference (Ideal value = 0): 0.032588705181255145
Max Equal Opportunity Difference (Ideal value = 0): 0.040506387751338635
Max Theil Index (Ideal value = 0): 0.324454002127635
Max Generalized Entropy Index (Ideal value = 0): 0.23128577805982717
Max Disparate Impact (Ideal value = 1): 1.0842130970886483

Ethnicity Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.7208942487863667
Max Average Odds Difference (Ideal value = 0): 0.705293972256414
Max Equal Opportunity Difference (Ideal value = 0): 0.7333376494961861
Max Theil Index (Ideal value = 0): 0.324454002127635
Max Generalized Entropy Index (Ideal value = 0): 0.23128577805982717
Max Disparate Impact (Ideal value = 1): 14.710765776280834

Gender+Ethnicity Bias Identification:
Max Statistical Parity Difference (Ideal value = 0): 0.5082888179295114
Max Average Odds Differ

In [131]:
# ── Fairness metrics for RW+CEOP (appended to metrics_values) ──
results_rw_ceop = evaluate_fairness(y, y_pred_all_classes_rw_ceop, X, algorithm='RW+CEOP')
results_rw_ceop


=== RW+CEOP ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
  RW+CEOP             Gender Overall (all classes)     0.614                0.008                            NaN                        NaN         2
  RW+CEOP             Gender             Non-COVID     0.614                0.008                          0.032                      0.041         2
  RW+CEOP             Gender                 Covid     0.614                0.008                          0.032                      0.041         2
  RW+CEOP          Ethnicity Overall (all classes)     0.614                0.078                            NaN                        NaN         5
  RW+CEOP          Ethnicity             Non-COVID     0.614                0.078                          0.739                      0.751         5
  RW+CEOP          Ethnicity                 Covid     0.614                0.078   

## Final Comparison

All metrics collected across the single and sequential mitigation algorithms,
in the same format as the Fairlearn notebook.

In [133]:
# =============================================================================
# EXPORT ALL RESULTS TO EXCEL
# -----------------------------------------------------------------------------
# For every case (Original Model + each mitigation algorithm + the sequential
# models) this cell collects:
#       Mean Accuracy, Accuracy STD,
#       Statistical Parity Difference, Average Odds Difference,
#       Equal Opportunity Difference, Theil Index,
#       Generalized Entropy Index, Disparate Impact
# for the three sensitive-feature settings used throughout the notebook
# (Gender, Ethnicity, Gender+Ethnicity) and writes them to one .xlsx workbook.
#
# The fairness numbers are the SAME max-over-(class x privileged class x group)
# values the metric cells above print, so the workbook should match them.
# =============================================================================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric

try:
    from tqdm.auto import tqdm
except ImportError:                       # tqdm is optional
    def tqdm(it, **kwargs):
        return it

EXCEL_PATH = 'Metrics_Values_AIF360.xlsx'      # <-- change the output path if you like

# Existing cells feed the raw multi-class argmax (0..3) into the binarised
# aif360 dataset. Keep False to reproduce exactly what those cells printed;
# set True for the stricter one-vs-rest comparison (pred == favourable class).
BINARIZE_PREDICTIONS = False


# -----------------------------------------------------------------------------
# 1. The cases: (name, in-memory variable names to try, CSV file fallback)
#    A case is skipped (with a message) if neither the variable nor the CSV
#    exists, so you can run this cell even after a partial run.
# -----------------------------------------------------------------------------
CASES = [
    ('Original Model',                  ['y_pred'],                                             None),
    ('Reweighing',                      ['y_pred_1'],                                           'y_pred_Reweighing.csv'),
    ('Disparate Impact Remover',        ['y_pred_2'],                                           'y_pred_DisparateImpactRemover.csv'),
    ('LFR',                             ['y_pred_3'],                                           'y_pred_LFR.csv'),
    ('Exponentiated Gradient Reduction',['y_pred_4'],                                           'y_pred_EGR.csv'),
    ('Grid Search Reduction',           ['y_pred_5'],                                           'y_pred_GSR.csv'),
    ('Equalized Odds Postprocessing',   [],                                                     'y_pred_EOP.csv'),
    ('Reject Option Classification',    [],                                                     'y_pred_ROC.csv'),
    ('Sequential: RW + EGR',            ['y_pred_rw_egr_labels', 'y_pred_rw_egr'],              'y_pred_Sequential_RW_EGR.csv'),
    ('Sequential: DIR + GSR',           ['y_pred_dir_gsr_labels', 'y_pred_dir_gsr'],            'y_pred_Sequential_DIR_GSR.csv'),
    ('Sequential: EGR + CEOP',          ['y_pred_egr_ceop_labels', 'y_pred_all_classes_egr_ceop'], 'y_pred_Sequential_EGR_CEOP.csv'),
    ('Sequential: RW + CEOP',           ['y_pred_rw_ceop_labels', 'y_pred_all_classes_rw_ceop'],   'y_pred_Sequential_RW_CEOP.csv'),
]
# NOTE: EOP and ROC both use the variable name `y_pred_final`, so only the last
# one survives in memory -> they are read from their CSVs instead.

# -----------------------------------------------------------------------------
# 2. Per-fold accuracies (optional).
#    Each section overwrites `scores`, so by the time you get here only the last
#    one is left. If you want the exact "Mean:" / "STD:" each section printed,
#    add a line like     CV_SCORES['Reweighing'] = list(scores)
#    right after that section's training cell. Anything not recorded falls back
#    to the overall out-of-fold accuracy (same value as the CV mean for equally
#    sized folds).
# -----------------------------------------------------------------------------
CV_SCORES = globals().get('CV_SCORES', {})


# -----------------------------------------------------------------------------
# 3. Sensitive-feature settings (identical to the metric cells above)
# -----------------------------------------------------------------------------
SENSITIVE_SETTINGS = [
    ('Gender',
     ['Gender'],
     [[[0]], [[1]]],
     [{'Gender': 0}, {'Gender': 1}]),

    ('Ethnicity',
     ['Ethnicity'],
     [[[0.0]], [[1.0]], [[2.0]], [[4.0]]],
     [{'Ethnicity': 0.0}, {'Ethnicity': 1.0}, {'Ethnicity': 2.0}, {'Ethnicity': 4.0}]),

    ('Gender + Ethnicity',
     ['Gender', 'Ethnicity'],
     [[[g], [e]] for g in [0, 1] for e in [0.0, 1.0, 2.0, 4.0]],
     [{'Gender': g, 'Ethnicity': e} for g in [0, 1] for e in [0.0, 1.0, 2.0, 4.0]]),
]

N_TARGET_CLASSES = N_CLASSES          # same range(N_CLASSES) the metric cells loop over


# -----------------------------------------------------------------------------
# 4. Helpers
# -----------------------------------------------------------------------------
def _safe_metric(fn):
    """Same guard used by the metric cells above."""
    try:
        val = fn()
        return val if np.isfinite(val) else 0.0
    except (TypeError, ZeroDivisionError, FloatingPointError):
        return 0.0


def _to_labels(obj):
    """Probabilities (n, n_classes) -> argmax; label vectors pass through."""
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        obj = obj.values
    arr = np.asarray(obj)
    if arr.ndim == 2 and arr.shape[1] > 1:
        arr = np.argmax(arr, axis=1)
    return arr.reshape(-1).astype(int)


def _load_case(var_names, csv_path):
    """Return (labels, source) from memory first, then from the saved CSV."""
    for name in var_names:
        if name in globals() and globals()[name] is not None:
            return _to_labels(globals()[name]), f'variable `{name}`'
    if csv_path and os.path.exists(csv_path):
        return _to_labels(pd.read_csv(csv_path)), csv_path
    return None, None


# Build the aif360 datasets once and reuse them for every case: they depend only
# on X / y, not on the predictions, and they are the slow part of the loop.
_full_df = pd.concat([X, y.rename('Status')], axis=1)
_dataset_cache = {}


def _get_datasets(protected_names, priv_classes, favorable_class):
    key = (tuple(protected_names), repr(priv_classes), favorable_class)
    if key not in _dataset_cache:
        ds = StandardDataset(
            df=_full_df,
            label_name='Status',
            favorable_classes=[favorable_class],
            protected_attribute_names=list(protected_names),
            privileged_classes=priv_classes,
        )
        ds.labels = ds.labels.astype(int)
        _dataset_cache[key] = (ds, ds.copy())
    return _dataset_cache[key]


def _max_metrics(pred_labels, protected_names, priv_class_lists, groups):
    """Max of each metric over target class x privileged class x group."""
    max_spd = max_aod = max_eod = max_theil = max_gei = max_di = -np.inf

    for favorable_class in range(N_TARGET_CLASSES):
        labels = ((pred_labels == favorable_class).astype(int)
                  if BINARIZE_PREDICTIONS else pred_labels)
        labels = labels.reshape(-1, 1).astype(int)

        for priv_classes in priv_class_lists:
            true_ds, pred_ds = _get_datasets(protected_names, priv_classes, favorable_class)

            if labels.shape[0] != true_ds.labels.shape[0]:
                raise ValueError(
                    f'{labels.shape[0]} predictions vs {true_ds.labels.shape[0]} rows '
                    'in the aif360 dataset - they must line up.')
            pred_ds.labels = labels

            for i, privileged_group in enumerate(groups):
                unprivileged_groups = groups[:i] + groups[i + 1:]
                metric = ClassificationMetric(
                    true_ds, pred_ds,
                    unprivileged_groups=unprivileged_groups,
                    privileged_groups=[privileged_group],
                )
                max_spd   = max(max_spd,   _safe_metric(metric.statistical_parity_difference))
                max_aod   = max(max_aod,   _safe_metric(metric.average_odds_difference))
                max_eod   = max(max_eod,   _safe_metric(metric.equal_opportunity_difference))
                max_theil = max(max_theil, _safe_metric(metric.theil_index))
                max_gei   = max(max_gei,   _safe_metric(metric.generalized_entropy_index))
                max_di    = max(max_di,    _safe_metric(metric.disparate_impact))

    return max_spd, max_aod, max_eod, max_theil, max_gei, max_di


# -----------------------------------------------------------------------------
# 5. Collect every case
# -----------------------------------------------------------------------------
y_true = np.asarray(y).reshape(-1).astype(int)
rows, skipped = [], []

for case_name, var_names, csv_path in tqdm(CASES, desc='Cases'):
    pred_labels, source = _load_case(var_names, csv_path)
    if pred_labels is None:
        skipped.append(case_name)
        continue
    if len(pred_labels) != len(y_true):
        skipped.append(f'{case_name} (length {len(pred_labels)} != {len(y_true)})')
        continue

    oof_accuracy = accuracy_score(y_true, pred_labels)
    folds = CV_SCORES.get(case_name)
    if folds is not None and len(folds):
        mean_acc, std_acc, acc_source = np.mean(folds), np.std(folds), 'CV folds'
    else:
        mean_acc, std_acc, acc_source = oof_accuracy, np.nan, 'out-of-fold'

    for feature_name, protected_names, priv_class_lists, groups in SENSITIVE_SETTINGS:
        spd, aod, eod, theil, gei, di = _max_metrics(
            pred_labels, protected_names, priv_class_lists, groups)

        rows.append({
            'Algorithm': case_name,
            'Sensitive Feature': feature_name,
            'Mean Accuracy': mean_acc,
            'Accuracy STD': std_acc,
            'Accuracy Source': acc_source,
            'Statistical Parity Difference': spd,
            'Average Odds Difference': aod,
            'Equal Opportunity Difference': eod,
            'Theil Index': theil,
            'Generalized Entropy Index': gei,
            'Disparate Impact': di,
            'Predictions From': source,
        })

results_df = pd.DataFrame(rows)

if skipped:
    print('Skipped (no predictions found):')
    for s in skipped:
        print('   -', s)

if results_df.empty:
    raise RuntimeError('Nothing to export - run the training cells first.')

# -----------------------------------------------------------------------------
# 6. Write the workbook: one long sheet, one sheet per sensitive feature,
#    plus an accuracy-only summary
# -----------------------------------------------------------------------------
METRIC_COLUMNS = [
    'Statistical Parity Difference', 'Average Odds Difference',
    'Equal Opportunity Difference', 'Theil Index',
    'Generalized Entropy Index', 'Disparate Impact',
]

accuracy_df = (results_df[['Algorithm', 'Mean Accuracy', 'Accuracy STD', 'Accuracy Source']]
               .drop_duplicates(subset='Algorithm')
               .reset_index(drop=True))

try:
    with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='All Results', index=False)
        accuracy_df.to_excel(writer, sheet_name='Accuracy', index=False)

        for feature_name, *_ in SENSITIVE_SETTINGS:
            sheet = results_df.loc[results_df['Sensitive Feature'] == feature_name,
                                   ['Algorithm', 'Mean Accuracy', 'Accuracy STD'] + METRIC_COLUMNS]
            if not sheet.empty:
                sheet.to_excel(writer, sheet_name=feature_name[:31], index=False)

        # readable column widths
        for worksheet in writer.book.worksheets:
            for column_cells in worksheet.columns:
                width = max(len(str(c.value)) if c.value is not None else 0
                            for c in column_cells)
                worksheet.column_dimensions[column_cells[0].column_letter].width = min(width + 2, 34)

    print(f'\nSaved {len(results_df)} rows to: {os.path.abspath(EXCEL_PATH)}')

except ImportError:
    fallback = EXCEL_PATH.replace('.xlsx', '.csv')
    results_df.to_csv(fallback, index=False)
    print(f'openpyxl is not installed (pip install openpyxl) - wrote {fallback} instead.')

results_df.round(4)

Cases: 100%|##########| 12/12 [00:10<00:00,  1.15it/s]

Saved 36 rows to: C:\Users\AMRashed\OneDrive - Shippensburg University\Dropbox\Nour\Nour\Paper Covid 19\Final Work\Journals\Computers in Biology and Medicine\Version-1\Metrics_Values_AIF360.xlsx
Out[101]: 
                           Algorithm  ...                   Predictions From
0                     Original Model  ...                  variable `y_pred`
1                     Original Model  ...                  variable `y_pred`
2                     Original Model  ...                  variable `y_pred`
3                         Reweighing  ...                variable `y_pred_1`
4                         Reweighing  ...                variable `y_pred_1`
5                         Reweighing  ...                variable `y_pred_1`
6           Disparate Impact Remover  ...                variable `y_pred_2`
7           Disparate Impact Remover  ...                variable `y_pred_2`
8           Disparate Impact Remover  ...    